# 01 · Can PointLLM recognise our scans?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/01_pointllm_recognition_test.ipynb)

**Goal:** feed our own iPhone scans (Scaniverse) to **PointLLM-7B** and record what it says. Everyday objects (chair, cup, …) show whether the pipeline works. Local objects (thela gari, kula, muura, …) are the real research question.

**What happens in this notebook**
1. Set up: GPU check, get the code, build PointLLM's Python environment
2. Load our scans from Google Drive and look at what the model will actually see
3. Start the model and ask questions, one at a time or in a batch → CSV
4. Optional tests, each changing one thing at a time:
   - **colour test**: same scans greyed out. Does PointLLM rely on colour or on shape?
   - **floor test**: same scans with the floor/desk plane removed. Was the floor confusing it?

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells top to bottom (Shift+Enter). The first run takes ~15–20 min (installs + a ~14 GB model download).


---
# Part 1 · Setup

## Check the GPU

`nvidia-smi` lists the graphics card. We need roughly 14 GB of GPU memory for the 7B model in float16; a T4 has 15 GB. `free -h` shows normal RAM.

> 🧠 Lines starting with `!` are not Python. They are run as shell (terminal) commands.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -h | head -2

## Get our project code

Colab only opened this *notebook*. Our helper code lives in the repo's `src/` folder, so we clone the repo into the Colab machine (or update it if it's already there).

> 🧠 **Python note:** `sys.path` is the list of folders Python searches when you write `import something`. Adding `src/` to it is what lets us `import pc_io` below.

In [ ]:
import os, sys

REPO_DIR = "/content/localpoint3d"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 https://github.com/Md-Mosabbir/localpoint3d.git {REPO_DIR}

sys.path.insert(0, f"{REPO_DIR}/src")
print("Code is in", REPO_DIR)

## Get PointLLM

This is the official PointLLM code. We use HTTPS because the `git@…` SSH link in their README doesn't work on Colab.

In [ ]:
POINTLLM_DIR = "/content/PointLLM"
if not os.path.isdir(POINTLLM_DIR):
    !git clone --depth 1 https://github.com/InternRobotics/PointLLM.git {POINTLLM_DIR}
else:
    print("PointLLM already cloned")

## Build a separate Python 3.10 environment

PointLLM was written for **Python 3.10 + PyTorch 2.0.1 + an old Transformers**. Colab's own Python is newer and those versions won't install into it. So we make a separate, isolated Python 3.10 environment (a *virtual environment*, "venv") at `/content/venv` using `uv`, a fast installer.

The notebook itself keeps running on Colab's Python. Later we start the model **inside the venv** as a separate background process.

> 🧠 `%%bash` at the top turns the whole cell into a shell script. `set -e` = stop at the first error. `if [ -d … ]` checks whether the folder already exists, so running the cell twice in one session doesn't fail.

In [ ]:
%%bash
set -e
pip install uv
if [ -d /content/venv ]; then
    echo "/content/venv already exists (this session ran the cell before), keeping it"
else
    uv venv --python 3.10 /content/venv
fi


## Install PointLLM's dependencies into that environment (~5 min)

- `torch==2.0.1` matches what PointLLM was tested with
- `-e .` installs the PointLLM repo itself plus everything in its requirements
- `plyfile`, `trimesh` are for reading our scan files (used by `src/pc_io.py`)
- `numpy<2` everywhere because the old libraries break with NumPy 2

In [ ]:
%%bash
set -e
cd /content/PointLLM
PY=/content/venv/bin/python
uv pip install --python $PY "torch==2.0.1" "numpy<2"
uv pip install --python $PY -e . "numpy<2"
uv pip install --python $PY accelerate plyfile trimesh matplotlib "numpy<2"

## Smoke test

Quick check that everything imports **before** we spend 10 minutes downloading model weights. You should see `imports OK`. Warnings about deprecated functions are fine.

In [ ]:
%%bash
cd /content/PointLLM
/content/venv/bin/python -c "
import torch, transformers, pointllm, plyfile, trimesh
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)
print('imports OK')"

---
# Part 2 · Our scans

## Install the file readers in Colab's own Python

The venv has them already. The notebook's Python needs them too, so we can preview scans here.

In [ ]:
!pip install plyfile trimesh

## Connect Google Drive

Our scans live on Google Drive. *Mounting* Drive makes it appear inside Colab as a normal folder, `/content/drive/MyDrive`. A Google popup asks for permission: allow it.

No Drive? Set `USE_DRIVE = False` and a file picker opens to upload scans from your computer instead (they disappear when the session ends).

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    os.makedirs("/content/scans", exist_ok=True)
    for name, data in files.upload().items():  # opens a file picker
        with open(os.path.join("/content/scans", name), "wb") as fh:
            fh.write(data)

## Point to your scans folder

Colab needs the folder's **path**, not a `drive.google.com/...` link. To get it:

1. Click the 📁 **Files** icon in the left sidebar
2. Open `drive` → `MyDrive` → your scans folder
3. Hover over the folder → **⋮** → **Copy path**
4. Paste it into `SCANS_DIR` below

Teammates: `Models` is shared from Md's Drive, so it is under *Shared with me* for you. On drive.google.com right-click it → *Organize* → *Add shortcut* → *My Drive*. Then the same path works for all of us.

Subfolders are searched too, so a layout like `Chair/Chair_1.ply`, `Cup/Cup_1.ply` works. OBJ files need their `.mtl` and texture image in the same folder, otherwise the colour is lost.

In [ ]:
import glob

SCANS_DIR = "/content/drive/MyDrive/Models"                 # <- our shared folder; paste another path to use a different one
RESULTS_DIR = "/content/drive/MyDrive/localpoint3d/results"  # where our CSVs get saved

if not USE_DRIVE:
    SCANS_DIR, RESULTS_DIR = "/content/scans", "/content/results"
if not os.path.isdir(SCANS_DIR):
    raise FileNotFoundError(f"No folder at {SCANS_DIR}. Copy the path from the Files panel (see above).")
os.makedirs(RESULTS_DIR, exist_ok=True)

EXTS = (".ply", ".obj", ".glb", ".gltf", ".off", ".stl")
# ** + recursive=True = also look inside subfolders (Chair/, Cup/, ...)
SCANS = sorted(p for p in glob.glob(f"{SCANS_DIR}/**/*", recursive=True) if p.lower().endswith(EXTS))
print(f"{len(SCANS)} scan(s) in {SCANS_DIR}")
for p in SCANS:
    print("  ", os.path.relpath(p, SCANS_DIR))

## What is in each file?

`describe()` (from `src/pc_io.py`) tells us the file type, how many points, whether it has colour, and its size.

- **Scaniverse** exports in metres, so a rickshaw should be about 2.5–3 m long.
- Scaniverse **Mesh** exports (`.ply` / `.obj`) are triangle meshes. We sample 100k points on the surface to turn them into a point cloud.
- `colour=no` on a mesh usually means its **texture image is missing**. A mesh `.ply` needs its `3DModel.jpg`, and an `.obj` needs its `.mtl` + image, in the same folder. Without it, the scan reaches PointLLM grey even in the "original" run.

In [ ]:
from pc_io import describe, parse_scan_name

for p in SCANS:
    try:
        print(describe(p))
    except Exception as e:
        print(os.path.basename(p), "-> could not read:", e)

## See what the model will see

PointLLM always gets exactly **8192 points**, centred and squashed into a unit sphere, with RGB colour. `preprocess()` does that:
1. load the file (sample the surface if it's a mesh)
2. pick 8192 well-spread points (*farthest point sampling*)
3. optionally rotate, then centre and scale
4. with `COLOUR = "grey"`, replace every colour with the same grey
5. with `FLOOR = "removed"`, delete the floor/desk plane first

**Orientation:** our `.ply` exports are **Z-up**, but our `.obj` exports are **Y-up** (the floor lies in the X-Z plane). So the two file types reach PointLLM in different orientations unless we rotate. `"-90 deg about X"` turns Z-up into Y-up, and `"+90 deg about X"` turns Y-up into Z-up.

**Colour:** `COLOUR = "grey"` shows the greyed-out version. Same points, only the colour is gone.

**Floor:** if a flat disc or plane is much bigger than the object, the object becomes a speck after scaling. Scans 4× or more bigger than their object were never recognised in our baseline. `FLOOR = "removed"` deletes the biggest *horizontal* plane near the bottom of the scan (RANSAC, `src/cleanup.py`). It knows which way is up from the file type: `.ply` Z-up, `.obj` Y-up. It removes floors and desks but not the rest of the room, so cropping in Scaniverse is still better.


In [ ]:
from IPython.display import display
from pointllm_helpers import preprocess, preview, ROTATIONS, PAPER_PROMPTS, COLOURS, FLOORS

ROTATION = "None"      # options: list(ROTATIONS)
COLOUR = "original"    # options: COLOURS = ["original", "grey"]
FLOOR = "kept"         # options: FLOORS = ["kept", "removed"]

for p in SCANS:
    pts, has_color = preprocess(p, rotation=ROTATION, colour=COLOUR, floor=FLOOR)
    note = "" if has_color else "  (file has no colour)"
    display(preview(pts, title=f"{os.path.basename(p)} · {COLOUR} · floor {FLOOR}{note}"))


---
# Part 3 · Ask PointLLM

## Start the model in the background

This starts `src/pointllm_worker.py` with the venv's Python 3.10. The worker loads the model **once** (the first time it also downloads ~14 GB) and then waits for questions. The helper `ask_pointllm()` sends it a question and waits for the answer.

**Randomness:** PointLLM picks a random starting point when it reads a point cloud, so without help the *same* scan can get a different answer each time. `seed` fixes that: the same seed always gives the same answer. Different seeds show how stable an answer is.

> 🧠 `subprocess.Popen` starts another program and keeps a "pipe" to it. We write questions into its `stdin` and read answers from its `stdout`, one JSON line each way.

In [ ]:
import json, subprocess

def start_worker():
    env = {**os.environ, "POINTLLM_DIR": POINTLLM_DIR}
    w = subprocess.Popen(["/content/venv/bin/python", "-u", f"{REPO_DIR}/src/pointllm_worker.py"],
                         cwd=POINTLLM_DIR, env=env, text=True, bufsize=1,
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("Loading the model (first time: downloads ~14 GB, be patient)...")
    for line in w.stdout:
        if "@@READY@@" in line:
            print("✅ Model ready.")
            return w
        if line.strip():
            print("  ", line.rstrip()[:200])
    raise RuntimeError("The worker stopped while loading. Read the log above.")


def ask_pointllm(path, prompt, rotation="None", colour="original", floor="kept", seed=0, sample=False):
    request = {"path": path, "prompt": prompt, "rotation": rotation, "colour": colour, "floor": floor,
               "seed": seed, "sample": sample}
    worker.stdin.write(json.dumps(request) + "\n")
    worker.stdin.flush()
    for line in worker.stdout:
        if line.startswith("@@RESULT@@"):
            res = json.loads(line[len("@@RESULT@@"):])
            if "error" in res:
                raise RuntimeError(res["error"])
            return res["answer"]
    raise RuntimeError("The worker died. Re-run this cell to restart it.")


if "worker" not in globals() or worker.poll() is not None:  # don't load twice
    worker = start_worker()

## Ask one question

Change `SCAN` and `QUESTION` (and `COLOUR` / `FLOOR` above) and re-run this cell as often as you like. The model stays loaded.

The three prompts from the PointLLM paper are in `PAPER_PROMPTS`:
- `"What is this?"`
- `"This is an object of "`: the model completes the sentence
- `"Caption this 3D model in detail."`

In [ ]:
SCAN = SCANS[0]
QUESTION = "What is this?"

print(os.path.basename(SCAN), "|", QUESTION)
print("PointLLM:", ask_pointllm(SCAN, QUESTION, rotation=ROTATION, colour=COLOUR, floor=FLOOR))

## Batch run: every scan × every prompt × every setting → CSV

This is the cell that produces our **results**. Each row records which file, object, capture method, rotation, colour, floor setting and prompt, and what PointLLM answered. The CSV goes to `RESULTS_DIR` (on Drive if you used Drive).

**Settings in this cell**
- `BATCH_SCANS`: which scans to run. All of them by default. To run only some, use e.g. `[p for p in SCANS if "Thela" in p]`.
- `ROTATIONS_TO_TRY`: does orientation change the answer?
- `COLOURS_TO_TRY`: does colour change the answer? `["original", "grey"]` runs every question twice: once with the scan's real colours, once all grey. The points are exactly the same both times, so a different answer means **colour** made the difference.
- `FLOORS_TO_TRY`: does the floor confuse the model? `["kept", "removed"]` runs every question with and without the floor plane.
- `SEEDS_TO_TRY`: PointLLM is a bit random inside. `[0]` = one fixed, repeatable answer per question. `[0, 1, 2]` asks 3 times with different randomness, so we can see whether an answer is stable or a coin flip (and take the majority).

Every list you add a second value to **doubles** the number of answers (three seeds triple it). Change one thing at a time.

**Time:** about 20 s per answer on a T4. 40 scans × 3 prompts × 2 colours = 240 answers ≈ 80 min. Start with a few scans.

The `file_has_colour` column matters: when it is `False`, the file had no colour, so its "original" run was already grey.


In [ ]:
import pandas as pd
from datetime import datetime
from pc_io import load_points

BATCH_SCANS = SCANS                      # e.g. [p for p in SCANS if "Thela" in p]
ROTATIONS_TO_TRY = [ROTATION]            # e.g. ["None", "-90 deg about X"]
COLOURS_TO_TRY = ["original", "grey"]    # ["original"] = no colour test
FLOORS_TO_TRY = ["kept"]                 # ["kept", "removed"] = floor test
SEEDS_TO_TRY = [0]                       # [0, 1, 2] = ask 3 times to see how stable answers are
PROMPTS_TO_TRY = PAPER_PROMPTS

rows = []
for p in BATCH_SCANS:
    meta = parse_scan_name(p)
    file_has_colour = load_points(p)[1] is not None
    for rot in ROTATIONS_TO_TRY:
        for col in COLOURS_TO_TRY:
            for fl in FLOORS_TO_TRY:
                for seed in SEEDS_TO_TRY:
                    for q in PROMPTS_TO_TRY:
                        ans = ask_pointllm(p, q, rotation=rot, colour=col, floor=fl, seed=seed)
                        rows.append({"file": os.path.relpath(p, SCANS_DIR), **meta, "file_has_colour": file_has_colour,
                                     "rotation": rot, "colour": col, "floor": fl, "seed": seed, "prompt": q, "answer": ans})
                        print(f"{os.path.basename(p)} | {rot} | {col} | floor {fl} | seed {seed} | {q!r}\n   -> {ans}\n")

results = pd.DataFrame(rows)
out_csv = f"{RESULTS_DIR}/pointllm_{datetime.now():%Y%m%d_%H%M}.csv"
results.to_csv(out_csv, index=False)
print("Saved", out_csv)
results


## Compare: one setting changed, answers side by side

For each test you ran with two values (colour: original vs grey, floor: kept vs removed) this shows one row per scan × prompt with both answers next to each other. `same = True` means changing that one setting did not change the answer.

> 🧠 `pivot` turns rows into columns: the two rows (e.g. original, grey) for one scan and prompt become one row with an `original` column and a `grey` column.


In [ ]:
SETTINGS = ["rotation", "colour", "floor", "seed"]

def side_by_side(results, setting, a, b):
    others = [s for s in SETTINGS if s != setting]   # everything except the setting we compare stays fixed
    side = results.pivot(index=["file", *others, "prompt"], columns=setting, values="answer").reset_index()
    side["same"] = side[a] == side[b]
    print(f"{setting}: {side['same'].sum()} of {len(side)} answers stayed exactly the same ({a} vs {b})")
    return side

for setting, a, b in [("colour", "original", "grey"), ("floor", "kept", "removed")]:
    if set(results[setting]) >= {a, b}:
        display(side_by_side(results, setting, a, b))
    else:
        print(f"{setting}: only one value was run, nothing to compare")


## (Optional) Free chat about one scan

Type questions. Press **Enter on an empty line** to stop.

In [ ]:
SCAN = SCANS[0]
print("Chatting about", os.path.basename(SCAN))
while True:
    q = input("You > ").strip()
    if not q:
        break
    print("PointLLM:", ask_pointllm(SCAN, q, rotation=ROTATION, colour=COLOUR, floor=FLOOR))

## Stop the model (frees the GPU)

Do this when you're done, or before switching to another model. Colab also frees everything when the runtime disconnects.

In [ ]:
worker.terminate()
print("Worker stopped.")

---
## Troubleshooting

| Problem | Try |
|---|---|
| `CUDA out of memory` | Runtime → Restart session, run again. Make sure no other worker is running. |
| Crash while loading, RAM full | System RAM ran out. Use a High-RAM runtime (Colab Pro) or retry. |
| Answers look random | Look at the preview. Is the object upright and are the background and ground cropped away? Try another `ROTATION`. |
| `unable to load image! … 3DModel.jpg` | That mesh's texture image isn't next to it on Drive, so its colour is lost. Upload the `.jpg` (or `.mtl` + image for OBJ) to the same folder. |
| `colour must be one of …` / `floor must be one of …` | `COLOUR` accepts `"original"` / `"grey"`, `FLOOR` accepts `"kept"` / `"removed"`. |
| `FLOOR = "removed"` deleted part of the object | The object is flatter than ~1 cm above the floor, or the up-axis guess is wrong for that file. Crop it in Scaniverse instead. |
| Worker error mentions an import | PointLLM's API changed. Compare `src/pointllm_helpers.py` with `PointLLM/pointllm/eval/PointLLM_chat.py`. |